# Task 1 — Inconsistencies and Duplicates

CSEN1095 Data Engineering — Task 1

## Part A — Look first
Load the original CSV and inspect its shape, data types, and categorical value counts before making any changes.

In [1]:
import pandas as pd

df = pd.read_csv('club_signups.csv')

print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)

categorical_columns = ['faculty', 'city', 'club', 'fee_paid']
for column in categorical_columns:
    print(f'\nValue counts for {column}:')
    print(df[column].value_counts(dropna=False))


Shape: (39, 9)

Dtypes:
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

Value counts for faculty:
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

Value counts for city:
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza    

## Part A — Normalize inconsistent values

The category columns are stripped and lower-cased mechanically first. Explicit dictionaries then map the remaining variants to the required canonical values.

In [2]:
clean = df.copy()

for column in ['faculty', 'city', 'club']:
    clean[column] = clean[column].astype('string').str.strip().str.lower()

faculty_map = {
    'met': 'MET',
    'media engineering and technology': 'MET',
    'iet': 'IET',
    'information engineering and technology': 'IET',
    'ems': 'EMS',
    'engineering and materials science': 'EMS',
    'pharmacy': 'Pharmacy',
    'pharma': 'Pharmacy',
    'management': 'Management'
}

city_map = {
    'cairo': 'Cairo',
    'giza': 'Giza',
    'el giza': 'Giza',
    'alexandria': 'Alexandria',
    'alex': 'Alexandria'
}

club_map = {
    'robotics': 'Robotics',
    'music': 'Music',
    'chess': 'Chess',
    'chess club': 'Chess',
    'football': 'Football',
    'soccer': 'Football',
    'debating': 'Debate',
    'debate club': 'Debate',
    'debate': 'Debate'
}

clean['faculty'] = clean['faculty'].map(faculty_map)
clean['city'] = clean['city'].map(city_map)
clean['club'] = clean['club'].map(club_map)

assert set(clean['faculty'].unique()) <= {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'}
assert set(clean['club'].unique()) <= {'Robotics', 'Debate', 'Music', 'Football', 'Chess'}
assert set(clean['city'].unique()) <= {'Cairo', 'Giza', 'Alexandria'}

print('Faculty values:', sorted(clean['faculty'].unique()))
print('Club values:', sorted(clean['club'].unique()))
print('City values:', sorted(clean['city'].unique()))


Faculty values: ['EMS', 'IET', 'MET', 'Management', 'Pharmacy']
Club values: ['Chess', 'Debate', 'Football', 'Music', 'Robotics']
City values: ['Alexandria', 'Cairo', 'Giza']


### Names and emails

In [3]:
clean['full_name'] = clean['full_name'].astype('string').str.split().str.join(' ').str.title()
clean['email'] = clean['email'].astype('string').str.strip().str.lower()

print(clean[['full_name', 'email']].head())


       full_name                             email
0      Hana Zaki      hana.zaki@student.guc.edu.eg
1     Ahmed Zaki     ahmed.zaki@student.guc.edu.eg
2   Mohamed Adel   mohamed.adel@student.guc.edu.eg
3   Mohamed Adel   mohamed.adel@student.guc.edu.eg
4  Laila Mostafa  laila.mostafa@student.guc.edu.eg


### Fee and timestamp conversion

In [4]:
fee_map = {
    'yes': True, 'y': True, 'true': True, '1': True,
    'no': False, 'n': False, 'false': False, '0': False
}
clean['fee_paid'] = (
    clean['fee_paid'].astype('string').str.strip().str.lower().map(fee_map)
)
assert clean['fee_paid'].dtype == bool

# Two timestamp formats occur. Values containing '/' use DD/MM/YYYY HH:MM.
raw_times = clean['signed_up_at'].astype('string')
slash_mask = raw_times.str.contains('/', regex=False)
parsed_times = pd.Series(index=clean.index, dtype='datetime64[ns]')
parsed_times.loc[slash_mask] = pd.to_datetime(
    raw_times.loc[slash_mask], format='%d/%m/%Y %H:%M'
)
parsed_times.loc[~slash_mask] = pd.to_datetime(
    raw_times.loc[~slash_mask], format='%Y-%m-%d %H:%M'
)
clean['signed_up_at'] = parsed_times
assert pd.api.types.is_datetime64_any_dtype(clean['signed_up_at'])

print('fee_paid dtype:', clean['fee_paid'].dtype)
print('signed_up_at dtype:', clean['signed_up_at'].dtype)


fee_paid dtype: bool
signed_up_at dtype: datetime64[ns]


## Part B — Duplicates

Exact duplicate submissions are removed before checking student/club duplicates.

In [5]:
exact_duplicates = clean.duplicated().sum()
print('Exact duplicate rows:', exact_duplicates)

clean = clean.drop_duplicates().copy()
rows_after_exact = len(clean)
print('Rows after exact duplicate removal:', rows_after_exact)


Exact duplicate rows: 3
Rows after exact duplicate removal: 36


### One row per student per club

In [6]:
# Sort chronologically so keep='last' retains the latest submission.
clean = clean.sort_values('signed_up_at')

rows_before_student_club = len(clean)
clean = clean.drop_duplicates(['student_id', 'club'], keep='last').copy()
student_club_removed = rows_before_student_club - len(clean)

assert not clean.duplicated(['student_id', 'club']).any()

print('Student+club rows removed:', student_club_removed)
print('Final rows:', len(clean))


Student+club rows removed: 4
Final rows: 32


### Required comparison: what if club spellings were not fixed first?

In [7]:
# This deliberately uses the original club spellings, after exact duplicates are removed.
raw_after_exact = df.drop_duplicates()
raw_club_duplicate_extras = raw_after_exact.duplicated(['student_id', 'club']).sum()

print('Duplicate rows found with raw club spellings:', raw_club_duplicate_extras)
print('Duplicate rows found after canonicalizing club spellings:', student_club_removed)


Duplicate rows found with raw club spellings: 2
Duplicate rows found after canonicalizing club spellings: 4


## Final checks and save

In [8]:
assert set(clean['faculty'].unique()) <= {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'}
assert set(clean['club'].unique()) <= {'Robotics', 'Debate', 'Music', 'Football', 'Chess'}
assert set(clean['city'].unique()) <= {'Cairo', 'Giza', 'Alexandria'}
assert clean['fee_paid'].dtype == bool
assert pd.api.types.is_datetime64_any_dtype(clean['signed_up_at'])
assert not clean.duplicated(['student_id', 'club']).any()

clean = clean.sort_values('signed_up_at').reset_index(drop=True)
clean.to_csv('club_signups_clean.csv', index=False)

print('Saved club_signups_clean.csv')
print('Final shape:', clean.shape)
print(clean.head())


Saved club_signups_clean.csv
Final shape: (32, 9)
  signup_id student_id      full_name                             email  \
0      S001    58-9118   Mariam Kamel   mariam.kamel@student.guc.edu.eg   
1      S002    55-2992   Mohamed Adel   mohamed.adel@student.guc.edu.eg   
2      S003    55-9245    Ali Ibrahim    ali.ibrahim@student.guc.edu.eg   
3      S004    61-7344  Laila Mostafa  laila.mostafa@student.guc.edu.eg   
4      S005    61-4733      Omar Adel      omar.adel@student.guc.edu.eg   

  faculty        city      club        signed_up_at  fee_paid  
0     IET        Giza  Robotics 2026-09-13 14:36:00      True  
1     IET  Alexandria    Debate 2026-09-13 15:38:00      True  
2     MET       Cairo     Chess 2026-09-13 21:16:00      True  
3     IET  Alexandria     Chess 2026-09-13 22:36:00      True  
4     EMS       Cairo  Football 2026-09-13 23:52:00      True  
